In [20]:
from sqlalchemy import create_engine, text
from sqlalchemy.orm import sessionmaker
import geopandas as gpd
import os
from dotenv import load_dotenv
from geoalchemy2.elements import WKBElement, WKTElement
load_dotenv()

True

### DATA BASE ENGINE

In [21]:
existing_database_url = os.getenv("BISELCO")
if existing_database_url:
    EXISTING_GIS_DATABASE= create_engine(existing_database_url)

current_database_url = os.getenv("BISELCOWEBSITE")
if current_database_url:
    CURRENT_GIS_DATABASE= create_engine(current_database_url)
    Session = sessionmaker(CURRENT_GIS_DATABASE)

In [22]:
# MUNICIPALITY DATA MIGRATION, NORMALIZATION AND PROCESSING
municipality = gpd.pd.read_sql(
    sql="""SELECT distinct municipality FROM gis.franchise_area order by municipality;""",
    con=EXISTING_GIS_DATABASE
)

values_mun = [(m,) for m in municipality['municipality'].to_list()]

# DATA MIGRATION FOR MUNICIPALITY VALUES
with Session.begin() as session:
    session.execute(statement=text(
        
        """INSERT INTO gis.municipality (name)
        VALUES (:mun)"""),
        params= [{"mun": m[0]} for m in values_mun] 
    )
    session.commit()
print("sucessfully inserted municipality")

sucessfully inserted municipality


In [23]:
# VILLAGE DATA MIGRATION, NORMALIZTION AND PROCESSING
village = gpd.pd.read_sql(
    sql = """SELECT distinct village, municipality FROM gis.franchise_area order by village;""",
    con=EXISTING_GIS_DATABASE
)

insert_stmt = text(
    """INSERT INTO gis.villages (municipality_id, name)
    VALUES (
        (SELECT id FROM gis.municipality WHERE name = :mun), :vill)"""
)
params = [{"mun": v[1], "vill": v[0]} for v in village.values]


with Session.begin() as session:
    session.execute(
        insert_stmt,
        params
    )
    session.commit()
print("sucessfully inserted village")

sucessfully inserted village


In [24]:
# BOUNDARY DATA MIGRATION, NORMALIZATION AND PROCESSING

boundary = gpd.read_postgis(
    sql = """SELECT distinct geom, municipality, village FROM gis.franchise_area;""",
    con=EXISTING_GIS_DATABASE,
    geom_col="geom"
)

params = [{"geom": b[0].wkt, "mun": b[1], "vill": b[2], "name": f"{b[2]} {b[1]}"} for b in boundary.values]

stmt = text(
    """INSERT INTO gis.boundary (geom, municipality_id, village_id, name)
    VALUES (:geom, (SELECT id FROM gis.municipality WHERE name = :mun), (SELECT id FROM gis.villages WHERE name = :vill and municipality_id = (SELECT id FROM gis.municipality WHERE name = :mun)), :name);
    """
)

with Session.begin() as session:
    session.execute(stmt, params)
    session.commit()
print("sucessfully inserted boundary")

sucessfully inserted boundary
